<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/security_camera_video_cleanup_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Security Camera Video Cleanup Notebook

This Colab notebook is tuned for **dark, noisy, low-detail night footage** like the sample frame you shared.

It gives you:
- upload + preview
- FFmpeg install
- metadata inspection
- a **recommended cleanup pass** for grainy nighttime footage
- a **stronger cleanup pass** if the first one is still ugly
- frame export for before/after comparison
- download commands for the finished file

## What this is tuned for
Your sample frame shows:
- heavy low-light noise
- soft detail / blur
- compression mush
- warm color cast
- weak contrast

So this notebook leans toward:
- denoising first
- gentle contrast fix
- modest sharpening
- optional upscale

Not miracle work. Just the least-bullshit recovery path.

In [ ]:
#@title 1) Install FFmpeg
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 1

In [ ]:
#@title 2) Upload your video
from google.colab import files
uploaded = files.upload()

if not uploaded:
    raise SystemExit('No file uploaded.')

INPUT_VIDEO = next(iter(uploaded.keys()))
print('Using input file:', INPUT_VIDEO)

In [ ]:
#@title 3) Inspect the file
import subprocess, json, shlex

def run(cmd):
    print('>>', cmd)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

probe_cmd = f'''ffprobe -v quiet -print_format json -show_format -show_streams {shlex.quote(INPUT_VIDEO)}'''
result = subprocess.run(probe_cmd, shell=True, text=True, capture_output=True)
meta = json.loads(result.stdout)
print(json.dumps(meta, indent=2)[:12000])

In [ ]:
#@title 4) Extract one preview frame before cleanup
import shlex
PRE_FRAME = 'frame_before.jpg'
!ffmpeg -y -i {shlex.quote(INPUT_VIDEO)} -vf "select=eq(n\,30)" -vframes 1 {PRE_FRAME}

from IPython.display import Image, display
display(Image(filename=PRE_FRAME))

## Recommended settings for your kind of footage

For the sample you showed, the safest first pass is:
- moderate denoise
- contrast/brightness correction
- tiny saturation bump
- modest sharpening
- 2x upscale with Lanczos

That avoids turning the video into crispy haunted oatmeal.

In [ ]:
#@title 5) Cleanup pass A (recommended first pass)
# Tuned for dark/grainy security-camera footage.

import shlex

OUTPUT_A = 'video_clean_pass_a.mp4'

vf = (
    'hqdn3d=1.8:1.8:6:6,'
    'eq=brightness=0.03:contrast=1.10:saturation=1.08,'
    'unsharp=5:5:0.7:5:5:0.0,'
    'scale=iw*2:ih*2:flags=lanczos'
)

cmd = (
    f'ffmpeg -y -i {shlex.quote(INPUT_VIDEO)} '
    f'-vf "{vf}" '
    f'-c:v libx264 -preset slow -crf 18 '
    f'-pix_fmt yuv420p -movflags +faststart '
    f'-c:a copy {shlex.quote(OUTPUT_A)}'
)

!{cmd}
print('Saved:', OUTPUT_A)

In [ ]:
#@title 6) Extract preview frame after pass A
POST_FRAME_A = 'frame_after_pass_a.jpg'
!ffmpeg -y -i video_clean_pass_a.mp4 -vf "select=eq(n\,30)" -vframes 1 {POST_FRAME_A}

from IPython.display import Image, display
display(Image(filename=POST_FRAME_A))

In [ ]:
#@title 7) Optional stronger cleanup pass B
# Use this only if pass A still looks too noisy.
# It may smooth detail a bit more.

import shlex

OUTPUT_B = 'video_clean_pass_b.mp4'

vf = (
    'hqdn3d=2.4:2.4:7:7,'
    'eq=brightness=0.04:contrast=1.13:saturation=1.06,'
    'unsharp=5:5:0.5:5:5:0.0,'
    'scale=iw*2:ih*2:flags=lanczos'
)

cmd = (
    f'ffmpeg -y -i {shlex.quote(INPUT_VIDEO)} '
    f'-vf "{vf}" '
    f'-c:v libx264 -preset slow -crf 18 '
    f'-pix_fmt yuv420p -movflags +faststart '
    f'-c:a copy {shlex.quote(OUTPUT_B)}'
)

!{cmd}
print('Saved:', OUTPUT_B)

In [ ]:
#@title 8) Optional non-upscaled version
# Useful if you want it cleaner without making the file bigger.

import shlex

OUTPUT_NO_UPSCALE = 'video_clean_no_upscale.mp4'

vf = (
    'hqdn3d=1.8:1.8:6:6,'
    'eq=brightness=0.03:contrast=1.10:saturation=1.08,'
    'unsharp=5:5:0.7:5:5:0.0'
)

cmd = (
    f'ffmpeg -y -i {shlex.quote(INPUT_VIDEO)} '
    f'-vf "{vf}" '
    f'-c:v libx264 -preset slow -crf 18 '
    f'-pix_fmt yuv420p -movflags +faststart '
    f'-c:a copy {shlex.quote(OUTPUT_NO_UPSCALE)}'
)

!{cmd}
print('Saved:', OUTPUT_NO_UPSCALE)

## If you need a little more brightness
You can bump the `eq=` filter values slightly.

Example:
- `brightness=0.05`
- `contrast=1.15`

Do **not** crank those too hard or the whole thing will look like radioactive soup.

In [ ]:
#@title 9) Custom tweak cell
# Change these values if you want to experiment.

import shlex

brightness = 0.03   #@param {type:"number"}
contrast   = 1.10   #@param {type:"number"}
saturation = 1.08   #@param {type:"number"}
luma_denoise = 1.8  #@param {type:"number"}
chroma_denoise = 1.8 #@param {type:"number"}
sharpen_amount = 0.7 #@param {type:"number"}
upscale_2x = True   #@param {type:"boolean"}

OUTPUT_CUSTOM = 'video_clean_custom.mp4'

vf_parts = [
    f'hqdn3d={luma_denoise}:{chroma_denoise}:6:6',
    f'eq=brightness={brightness}:contrast={contrast}:saturation={saturation}',
    f'unsharp=5:5:{sharpen_amount}:5:5:0.0'
]

if upscale_2x:
    vf_parts.append('scale=iw*2:ih*2:flags=lanczos')

vf = ','.join(vf_parts)

cmd = (
    f'ffmpeg -y -i {shlex.quote(INPUT_VIDEO)} '
    f'-vf "{vf}" '
    f'-c:v libx264 -preset slow -crf 18 '
    f'-pix_fmt yuv420p -movflags +faststart '
    f'-c:a copy {shlex.quote(OUTPUT_CUSTOM)}'
)

!{cmd}
print('Saved:', OUTPUT_CUSTOM)

In [ ]:
#@title 10) Compare a frame from custom output
POST_FRAME_CUSTOM = 'frame_after_custom.jpg'
!ffmpeg -y -i video_clean_custom.mp4 -vf "select=eq(n\,30)" -vframes 1 {POST_FRAME_CUSTOM}

from IPython.display import Image, display
display(Image(filename=POST_FRAME_CUSTOM))

In [ ]:
#@title 11) Download your finished video(s)
from google.colab import files

# Uncomment whichever file you want to download.
files.download('video_clean_pass_a.mp4')
# files.download('video_clean_pass_b.mp4')
# files.download('video_clean_no_upscale.mp4')
# files.download('video_clean_custom.mp4')

## My recommendation for your sample

Start with **Pass A**.

If it still looks too noisy, try **Pass B**.

If Pass B starts making things look too smeared, back off and use **Pass A** or the **custom tweak cell**.

For this specific kind of night camera footage, trying to over-sharpen it is how people accidentally create cursed raccoon-vision. Keep it moderate.